In [2]:
import pandas as pd
import numpy as np
from FinMind.data import DataLoader
from pandas import api
import requests
import time
from pprint import pprint
from tqdm import tqdm
from loguru import logger
import sys
import os
from contextlib import contextmanager
import pyppeteer
import pickle


In [3]:
DATA_LOADER = DataLoader()
DATA_LOADER.login(user_id="Damingwang0007" , password="gadhyJ-vakfas-1tarry")

2026-07-16 13:36:56.484 | INFO     | FinMind.data.finmind_api:login:91 - Login success


True

In [ ]:
def get_stock_data(stock_id: str, start_date: str = '2026-07-01', dl: DataLoader = None) -> tuple[pd.DataFrame, float]:
       """
       取得股票歷史價格與總發行股數資料，供計算部位成本分佈使用
       參數:
              stock_id: str, 股票代碼 (例如 '2330')
              start_date: str, 資料起始日期 (格式 'YYYY-MM-DD')
       回傳:
              tuple: (歷史價格 DataFrame, 總發行股數)
       """
       if dl is None:
              dl = DataLoader()
              dl.login(user_id="Damingwang0007" , password="gadhyJ-vakfas-1tarry")

       # 2. Fetch the Foreign Shareholding data
       
       df = dl.taiwan_stock_shareholding(
              stock_id=stock_id,
              start_date="2024-04-01",
       )
       latest_record = df.iloc[-1]
       total_shares = latest_record['NumberOfSharesIssued']
       data = dl.taiwan_stock_daily(stock_id=stock_id, start_date=start_date)
       return data, total_shares

In [ ]:
import os
from datetime import datetime, timedelta
import pandas as pd
from tqdm import tqdm

def update_or_download_stock(stock_id: str, dl: DataLoader) -> None:
    file_path = f'../data/{stock_id}_stock_data.csv'
    
    # 預設初始下載日期（如果 CSV 不存在時使用）
    default_start_date = '2023-01-01' 
    
    # 1. 讀取現有 CSV 檔案
    if os.path.exists(file_path):
        try:
            existing_df = pd.read_csv(file_path)
            # 確保日期欄位為 datetime 格式，方便排序與計算
            existing_df['date'] = pd.to_datetime(existing_df['date'])
            
            # --- 選擇起始日期方式（二選一） ---
            
            # 【推薦方式】自動接續最新日期（避免放假或忘記執行導致資料漏掉）
            latest_date = existing_df['date'].max()
            start_date = (latest_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
            
            # 【強制最近一週方式】（若堅持只要最新 7 天，請取消註解下方兩行）
            # one_week_ago = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
            # start_date = one_week_ago
            
        except Exception as e:
            print(f"讀取 {stock_id} CSV 失敗，將重新下載。錯誤: {e}")
            existing_df = pd.DataFrame()
            start_date = default_start_date
    else:
        existing_df = pd.DataFrame()
        start_date = default_start_date

    # 2. 下載新資料
    try:
        # 下載最新一週（或接續）的日線資料
        new_data = dl.taiwan_stock_daily(stock_id=stock_id, start_date=start_date)
        
        if new_data is None or new_data.empty:
            print(f"[{stock_id}] 沒有更新的資料。")
            return
        
        new_data['date'] = pd.to_datetime(new_data['date'])
        
        # 3. 合併新舊資料
        if not existing_df.empty:
            # 拼接資料
            combined_df = pd.concat([existing_df, new_data], ignore_index=True)
            # 去除重複的日期（保留最後下載的，以防有盤後資料修正）
            combined_df.drop_duplicates(subset=['date'], keep='last', inplace=True)
        else:
            combined_df = new_data
            
        # 排序日期
        combined_df.sort_values(by='date', inplace=True)
        
        # 將日期轉回字串格式，保持 CSV 乾淨好讀
        combined_df['date'] = combined_df['date'].dt.strftime('%Y-%m-%d')
        
        # 4. 寫回 CSV 檔案
        # 建立資料夾（防呆）
        os.makedirs('../data', exist_ok=True)
        combined_df.to_csv(file_path, index=False)
        
    except Exception as e:
        print(f"更新 {stock_id} 資料時發生錯誤: {e}")

In [5]:
stock_ids = ['1216', '1303', '2059', '2301', '2303', '2308', '2317', '2327', '2330', '2344','2345', '2357', '2360', '2368', '2382', '2383', '2395', '2408', '2412', '2449','2454', '2603', '2880', '2881', '2882', '2883', '2884', '2885', '2886', '2887','2890', '2891', '2892', '3008', '3017', '3037', '3045', '3231', '3443', '3653','3661', '3665', '3711', '4904', '4958', '5880', '6505', '6669', '7769', '8046']

In [7]:
stock_ids = [
       '1101', '1102', '1301', '1326', '1402', '1476', '1503', '1504', '1513', '1519',
       '1560', '1590', '1605', '1717', '2105', '2207', '2312', '2313', '2324', '2337',
       '2347', '2353', '2354', '2356', '2371', '2376', '2377', '2379', '2385', '2404',
       '2409', '2455', '2474', '2492', '2542', '2609', '2610', '2615', '2618', '2633',
       '2645', '2801', '2812', '2834', '2912', '3005', '3023', '3034', '3036', '3044'
       ]

In [8]:
# 初始化 DataLoader (只登入一次，避免重複登入)
if 'DATA_LOADER' not in locals():
    DATA_LOADER = DataLoader()
    DATA_LOADER.login(user_id="Damingwang0007" , password="gadhyJ-vakfas-1tarry")

# 批次更新所有股票 CSV
for stock_id in tqdm(stock_ids, desc="Updating stock database"):
    update_or_download_stock(stock_id, DATA_LOADER)

Updating stock database:   2%|▏         | 1/50 [00:00<00:14,  3.44it/s]2026-07-16 13:39:21.468 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 1102
2026-07-16 13:39:21.542 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 1301
Updating stock database:   6%|▌         | 3/50 [00:00<00:06,  7.77it/s]2026-07-16 13:39:21.611 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 1326
2026-07-16 13:39:21.711 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 1402
Updating stock database:  10%|█         | 5/50 [00:00<00:04,  9.36it/s]2026-07-16 13:39:21.784 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 1476
2026-07-16 13:39:21.865 | INFO     | FinMind.data.finmind_api:get_data:171 - download Dataset.TaiwanStockPrice, data_id: 1503
Updating stock database:  14%|█